# Fairness in Automated Decision Systems

**Initial Due Date: 2026-01-16 10:00AM**  
**Final Due Date: 2026-01-19 4:15PM**

## Learning Objectives

By the end of this activity, you will be able to:

-   Mathematically describe the concepts of error rate parity and sufficiency as competing notions of fairness in automated decision systems.
-   Prove that error rate parity and sufficiency cannot both be satisfied when base rates differ between groups.
-   Reason about how to approximately enforce sufficiency as a fairness criterion by adjusting decision thresholds for different groups.

## Introduction

In [class](https://middcs.github.io/data-science-notes/source/26-compas.html), we introduced the COMPAS algorithm, which predicts the likelihood of recidivism for defendants based on various features and which was the subject of major scholarly controversy concerning whether or not the algorithm was biased. In this set of notes, we’ll further explore the tensions between competing concepts of fairness in the context of the COMPAS dataset.

### Part A: Error Rate Parity and Sufficiency

Recall from lecture that error rate parity and sufficiency are two competing conceptions of fairness:

> **Error rate parity** requires that the false positive rate and false negative rate be equal across groups. **Sufficiency** requires that a given prediction “means the same thing” across groups, in the sense that defendants who receive the same prediction should indeed be equally risky.

To make these concepts more concrete, let’s define:

-   $\mathrm{TP}$, the count of true positives in the data (i.e. defendants who were predicted to recidivate and did so).
-   $\mathrm{TN}$, the count of true negatives in the data (i.e. defendants who were predicted not to recidivate and did not do so).
-   $\mathrm{FP}$, the count of false positives in the data (i.e. defendants who were predicted to recidivate but did not do so).
-   $\mathrm{FN}$, the count of false negatives in the data (i.e. defendants who were predicted not to recidivate but did so).

As we have previously discussed, the *false positive rate* has formula

$$
\begin{aligned}
    \mathrm{FPR} = \frac{FP}{FP + TN}
\end{aligned}
$$

and the *false negative rate* has formula

$$
\begin{aligned}
    \mathrm{FNR} = \frac{FN}{FN + TP}\;.
\end{aligned}
$$

The principle of error-rate parity says that these quantities should be the same for every group for a fair classifier.

To operationalize the idea of sufficiency, we define the *positive predictive value*:

$$
\begin{aligned}
    \mathrm{PPV} = \frac{TP}{TP + FP}\;.
\end{aligned}
$$

This is the fraction of the time that a positive prediction is correct. The principle of sufficiency says that this quantity should be the same for every group.

Can we achieve both error rate parity and sufficiency in the COMPAS data set? Surprisingly, we can actually address this as a *math* question.

Let $p$ be the prevalence of recidivism; that is, the fraction of defendants who actually recidivate.

#### Exercise A.1

Give a formula for $p$ in terms of $\mathrm{TP}$, $\mathrm{TN}$, $\mathrm{FP}$, and $\mathrm{FN}$. Express your answer using the Notebook’s LaTeX math typesetting (i.e. using double dollar signs `$$` to open and close math mode). Fractions can be implemented using the `\frac{numerator}{denominator}` command in math mode.

*[TODO: Your response here]*

#### Exercise A.2

The following expression relates the prevalence $p$ to the error rates and positive predictive value:

$$
\begin{aligned}
    p = \frac{1}{1 + \frac{\mathrm{TPR}}{\mathrm{FPR}}\frac{1- \mathrm{PPV}}{\mathrm{PPV}}}
\end{aligned}
$$

For the interested reader, this relation can be derived as follows:

$$
\begin{aligned}
    \frac{1}{1 + \frac{\mathrm{TPR}}{\mathrm{FPR}}\frac{1- \mathrm{PPV}}{\mathrm{PPV}}} &= \frac{1}{1 + \frac{\frac{\mathrm{TP}}{\mathrm{FN} + \mathrm{TP}}}{\frac{\mathrm{FP}}{\mathrm{FP} + \mathrm{TN}}}\frac{1- \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}}{\frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}}}  \\ 
    &= \frac{1}{1 + \frac{\frac{\mathrm{TP}}{\mathrm{FN} + \mathrm{TP}}}{\frac{\mathrm{FP}}{\mathrm{FP} + \mathrm{TN}}}\frac{\frac{\mathrm{FP}}{\mathrm{TP} + \mathrm{FP}}}{\frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}}}  \\
    &= \frac{1}{1 + \frac{\mathrm{TP} (\mathrm{FP} + \mathrm{TN}) \mathrm{FP}}{\mathrm{FP} (\mathrm{FN} + \mathrm{TP}) \mathrm{TP}}}  \\
    &= \frac{1}{1 + \frac{\mathrm{FP} + \mathrm{TN}}{\mathrm{FN} + \mathrm{TP}}}  \\
    &= \frac{1}{\frac{\mathrm{FN} + \mathrm{TP} + \mathrm{FP} + \mathrm{TN}}{\mathrm{FN} + \mathrm{TP}}}  \\
    &= \frac{\mathrm{FN} + \mathrm{TP}}{\mathrm{TP} + \mathrm{TN} + \mathrm{FP} + \mathrm{FN}}  \\
    &= p\;.
\end{aligned}
$$

Recall from class that the prevalences of recidivism are different in this data set between white and Black defendants (i.e., $p_a \neq p_b$). With this and the relation above in mind, is it possible to achieve both error rate parity ($\mathrm{TPR}_a == \mathrm{TPR}_b$ and $\mathrm{FPR}_a == \mathrm{FPR}_b$) and sufficiency ($\mathrm{PPV}_a == \mathrm{PPV}_b$) simultaneously in this data set? Explain your reasoning?

*[TODO: Your response here]*

## Part B: Fairness by Sex

Let’s apply some of the techniques we’ve studied in this class to analyze the fairness of the COMPAS classifier by defendant sex rather than race.

The code block below loads and prepares the COMPAS dataset as we did in class.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
np.set_printoptions(precision = 3)
pd.set_option('display.precision', 3)

url = "https://github.com/propublica/compas-analysis/raw/master/compas-scores-two-years.csv"
compas = pd.read_csv(url)

cols = ["sex", "race", "decile_score", "two_year_recid"]
compas = compas[cols]

# Create a binary predicted recidivism variable using a threshold of 4
compas["predicted_recid"] = (compas["decile_score"] > 4).astype(int)

compas.head()

### Exercise B1

Compute the prevalence of recidivism by sex in this data set and print the result (that is, you should compute two prevalences, one among female defendants and one among male defendants). Based on your results, do you expect the COMPAS predictions to simultaneously satisfy both error rate parity and sufficiency by sex? Please save your result as `B1_prevalence_by_sex,` and then print it.

***Specifications***: *Your result should be a `pandas.Series` with index corresponding to `sex`, and with values corresponding to the prevalences. Use idiomatic Pandas code to compute the prevalences, with no for-loops.*

In [ ]:
# TODO: Your code here
print(B1_prevalence_by_sex)

*[TODO: Your response here]*

### Exercise B2

Compute the false positive rate (FPR) and false negative rate (FNR) by sex (so, you should have four numbers in total). Please save your results as two separate `pandas.Series` objects, `B2_fpr_by_sex` and `B2_fnr_by_sex`, and then print them.

Then, determine whether the COMPAS predictions approximately satisfy error rate parity by sex.

***Specifications***: *For the FPR, your result should be a `pandas.Series` with index corresponding to `sex`, and with values corresponding to the false positive rates. The TPR should have the same structure. Please use idiomatic Pandas code to compute the prevalences, with no for-loops.*

In [ ]:
# TODO: Your code here

print("False positive rates")
print(B2_fpr_by_sex)

print("\nFalse negative rates")
print(B2_fnr_by_sex)

*[TODO: Your response here]*

### Exercise B3: Sufficiency

Compute the positive predictive value (PPV) by sex, and save your result to a variable `B3_ppv_by_sex`. Then, determine whether the COMPAS predictions approximately satisfy sufficiency by sex.

***Specifications***: *Your result should be a `pandas.Series` with index corresponding to `sex`, and with values corresponding to the positive predictive values.Please use idiomatic Pandas code to compute the prevalences, with no for-loops.*

In [ ]:
# TODO: Your code here

print("Positive predictive values")
print(B3_ppv_by_sex)

*[TODO: Your response here]*

## Part C: Calibration

Suppose that we want to modify the way we use COMPAS scores to make decisions in such a way that achieves sufficiency – that is, our aim is to have approximately equal positive predictive values across groups. One way to do this is to change the threshold at which we classify a defendant as high risk. For example, we might consider a male defendant to be high risk if their decile score is greater than 6, while we might consider a female defendant to be high risk if their decile score is greater than 3.

### Exercise C1

Write code to compute the positive predictive value for both sexes at all possible combinations of thresholds between 0 and 9, inclusive, for male and female defendants separately. Further compute the absolute difference in positive predictive values between groups. Store your results in a DataFrame named `df` with columns `t_male`, `t_female`, `ppv_male`, `ppv_female`, and `ppv_diff`, where `t_male` and `t_female` are the thresholds for male and female defendants respectively, `ppv_male` and `ppv_female` are the corresponding positive predictive values, and `ppv_diff` is the absolute difference in positive predictive values between groups.

It is absolutely OK in this instance to use one or more for-loops to iterate over the combinations of thresholds (although approaches that don’t use explicit for loops are also possible). You can accumulate a DataFrame row-by-row using this pattern:

``` python
df = pd.DataFrame()
for i in range(10): 
    df = pd.concat([df, pd.DataFrame({"col1": [i], "col2": [i**2]})], ignore_index=True)

# now df contains two columns, col1 and col2, with 10 rows
```

In [ ]:
# TODO: Your code here

### Exercise C2

Suppose now that we are willing to accept a difference in PPVs of up to 0.03 between male and female defendants. Create a scatterplot showing the possible combinations of thresholds which achieve this. Place `ppv_male` on the x-axis and `ppv_female` on the y-axis, with a point to represent that a given combination of thresholds achieves this level of sufficiency. Comment on the feasibility of achieving sufficiency by sex in this way.

In [ ]:
# TODO: Your code here
plt.show()

*[TODO: Your response here]*

## Part D

Medianbury College uses a unique admissions mechanism in which *only* the SAT score is used to make admissions decisions: all students who score above a threshold are admitted. However, news media has recently reported that the college uses *different* thresholds for female and male students. In particular, a higher threshold is used for female students. In an interview, a spokesperson for the college stated that the college developed this policy by studying the relevance of the SAT as a predictor of academic success in college as measured by GPA.

> #### ⚖️ Is that legal?
>
> We Are Not Lawyers™, but we understand the formal use of such a policy to be *illegal* under U.S. law. Here, we’re going to ask about definitions of fairness independent of legal considerations.

### Exercise D1

Write a brief paragraph in which you argue that Medianbury’s use of different admissions thresholds is necessarily unfair. Please include discussion of error rate parity and sufficiency in your response.

*[TODO: Your response here]*

### Exercise D2

Write a brief paragraph in which you defend Medianbury’s use of different admissions thresholds as a justifiable way to pursue fairness. Please include discussion of error rate parity and sufficiency in your response. What would have to be true about the predictive relationship between SAT scores and GPA for your defense to hold?

*[TODO: Your response here]*

## Collaboration statement

In a new text cell immediately below this paragraph (or by editing this text cell to add a paragraph), briefly list who or what you collaborated with and how. Cite any sources here or with relevant inline comments in your code. Acknowledge all contributors, both people and AI, and what portions of this notebook they contributed. You do not need to cite or acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant assignment on [Gradescope](https://gradescope.com) via the “Upload option” (guide [here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)). **Both files must be uploaded at the same time and the file names must match the specification exactly for the autotesting to run successfully.**

1.  `activity_compas.ipynb`: Your completed IPython notebook. You can obtain this via the “File→Download→Download .ipynb” menu option in Colab.
2.  `activity_compas.py`: Your completed IPython notebook as a Python file. You can obtain this via the “File→Download→Download .py” menu option in Colab. This file is used to provide line-level feedback on your submission.

You can submit multiple times, with only the most recent submission (before the final due date) assessed for credit. Gradescope will run a series of automated unit tests on your notebook (which may takes 10s of seconds depending on the complexity of the notebook). Note that the tests performed by Gradescope are limited. Passing all of the visible tests does not guarantee that your submission correctly satisfies all of the requirements of the assignment.